In [5]:
# G4 — jalankan penanganan pada data tim dan periksa nilai k
# Salin pola Bagian E dan F, ganti dataset dan nama kolomnya.
# Simpan hasilnya ke data/processed/ dan JANGAN unggah data/raw/

import pandas as pd
import hashlib
import numpy as np
import os

# Memuat dataset mentah dari folder data/raw/ (pastikan data mentah tidak di-commit)
FILE = 'tourism_dataset.csv'

if not os.path.exists(FILE):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        raise FileNotFoundError('Letakkan CSV di folder yang sama dengan notebook ini.')

df = pd.read_csv(FILE)

# Membuat salinan working dataframe
aman_tour = df.copy()

# 1. Penanganan kolom 'Location' (dianggap sebagai pengenal langsung / ID spesifik)
GARAM_TOUR = 'rahasia-tim-pariwisata-2025'

def samarkan_lokasi(nilai, garam):
    if pd.isna(nilai):
        return np.nan
    gabung = str(garam) + str(nilai)
    return hashlib.sha256(gabung.encode('utf-8')).hexdigest()[:12]

aman_tour['Location'] = aman_tour['Location'].apply(lambda x: samarkan_lokasi(x, GARAM_TOUR))

# 2. Pemeriksaan k-anonymity (misalnya menggunakan kombinasi: Country, Category, Accommodation_Available)
def cek_k(data, kolom):
    grup_counts = data.groupby(kolom, observed=True).size()
    k = grup_counts.min() if len(grup_counts) > 0 else 0
    grup_1_orang = (grup_counts == 1).sum()
    total_grup = len(grup_counts)
    return pd.Series({'k': k, 'grup_1_orang': grup_1_orang, 'total_grup': total_grup})

quasi_cols = ['Country', 'Category', 'Accommodation_Available']
print("Hasil Cek k-Anonymity:")
print(cek_k(aman_tour, quasi_cols))

# 3. Menyimpan hasil ke folder data/processed/
os.makedirs('data/processed', exist_ok=True)
aman_tour.to_csv('data/processed/tourism_clean.csv', index=False)
print("\nData bersih berhasil disimpan ke data/processed/tourism_clean.csv")


Saving tourism_dataset.csv to tourism_dataset.csv
Hasil Cek k-Anonymity:
k               51
grup_1_orang     0
total_grup      84
dtype: int64

Data bersih berhasil disimpan ke data/processed/tourism_clean.csv
